# Entrenamiento y versionado de modelos

In [17]:
from __future__ import annotations

import argparse
import json
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Callable

import numpy as np
import optuna
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from preprocess import ROOT, RUTA_FINAL, TARGET, huella, construir_pipeline

SEMILLA = 42
RUTA_LOGS = ROOT / "results" / "experiment_logs.csv"

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Espacios de búsqueda

In [18]:
def _mlp(**params) -> TransformedTargetRegressor:
    """Red neuronal (MLP) con el target estandarizado.

    El target está en kW (~10000) y los pesos de la red arrancan cerca de 0: sin estandarizarlo
    la red entrena muy mal. El escalado de y se ajusta solo con los datos de entrenamiento que
    recibe .fit() y viaja dentro del objeto, así que .predict() devuelve kW directamente."""
    return TransformedTargetRegressor(regressor=MLPRegressor(**params), transformer=StandardScaler())


# Optuna no acepta tuplas como opciones categóricas: se elige un nombre y se traduce.
_CAPAS_MLP = {"32": (32,), "64": (64,), "64-32": (64, 32), "128-64": (128, 64)}


def _espacio_ridge(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e4, log=True)}


def _espacio_lasso(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e3, log=True)}


def _espacio_rf(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 100, 500, step=100),
        "max_depth": t.suggest_categorical("max_depth", [None, 4, 8, 12]),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
        "max_features": t.suggest_float("max_features", 0.3, 1.0),
    }


def _espacio_xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 6),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1.0, 100.0, log=True),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 5),
    }


def _espacio_mlp(t: optuna.Trial) -> dict:
    return {
        "hidden_layer_sizes": _CAPAS_MLP[t.suggest_categorical("hidden_layer_sizes", list(_CAPAS_MLP))],
        "activation": t.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": t.suggest_float("alpha", 1e-4, 10.0, log=True),   # regularización L2
        "learning_rate_init": t.suggest_float("learning_rate_init", 1e-3, 1e-2, log=True),
        "batch_size": t.suggest_categorical("batch_size", [32, 64, 128]),
    }

In [19]:
ESPACIOS: dict[str, dict] = {
    "ridge": {"clase": Ridge, "fijos": {"random_state": SEMILLA}, "espacio": _espacio_ridge},
    "lasso": {"clase": Lasso, "fijos": {"random_state": SEMILLA, "max_iter": 100_000},
              "espacio": _espacio_lasso},
    "random_forest": {"clase": RandomForestRegressor, "fijos": {"random_state": SEMILLA, "n_jobs": -1},
                      "espacio": _espacio_rf},
    "xgboost": {"clase": XGBRegressor,
                "fijos": {"random_state": SEMILLA, "n_jobs": -1, "tree_method": "hist"},
                "espacio": _espacio_xgb},
    "mlp": {
        "clase": _mlp,
        "rfe": False,   # TransformedTargetRegressor no expone coef_/feature_importances_: RFE no aplica
        # Sin early_stopping: scikit-learn lo haría con un 10 % ALEATORIO de train, que en una
        # serie temporal mezcla pasado y futuro. Corta por n_iter_no_change sobre la pérdida de train.
        "fijos": {"random_state": SEMILLA, "solver": "adam", "max_iter": 1000,
                "n_iter_no_change": 20, "early_stopping": False},
        "espacio": _espacio_mlp,
    },
}

## Datos

In [20]:
def cargar_datos(ruta: str | Path = RUTA_FINAL, incluir_test: bool = False) -> pd.DataFrame:
    """Lee dataset_final.csv indexado por fecha. Por defecto descarta las filas de test:
    ni siquiera quedan en memoria. Pasar incluir_test=True solo para la evaluación final."""
    df = pd.read_csv(ruta, parse_dates=["fecha"]).sort_values("fecha").set_index("fecha")
    if not incluir_test:
        df = df[df["split"] != "test"]
    return df


def particion(df: pd.DataFrame, split: str) -> tuple[pd.DataFrame, pd.Series]:
    """(X, y) de un split. Features = todo menos el target y la columna `split`."""
    d = df[df["split"] == split]
    return d.drop(columns=[TARGET, "split"]), d[TARGET]


def _metricas(y, pred) -> dict:
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    ok = ~np.isnan(pred)  # los primeros días de un rolling quedan en NaN
    y, pred = y[ok], pred[ok]
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y, pred))),
        "r2": float(r2_score(y, pred)),
    }

## Baselines

In [21]:
def predicciones_baselines(df: pd.DataFrame) -> pd.DataFrame:
    """Predicción de cada baseline para la fila D (que apunta al consumo de D+1).

    El target de la fila D es frio(D+1), así que frio(D) = target de la fila D-1:
      persistencia   : frio(D)                      = y.shift(1)
      media móvil 7d : promedio de frio(D-6 .. D)   = y.shift(1).rolling(7).mean()
    Solo usa información disponible en D: no hay fuga hacia el futuro."""
    if not (df.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError("Hay días faltantes en dataset_final.csv: shift(1) ya no equivale a 'ayer'.")
    y = df[TARGET]
    return pd.DataFrame({
        "baseline_persistencia": y.shift(1),
        "baseline_media_movil_7d": y.shift(1).rolling(7).mean(),
    })

## Registro de experimentos

In [22]:
COLUMNAS_LOG = [
    "corrida", "timestamp", "modelo",
    "etapa",             # baseline | optuna_cv | optuna_val
    "estado",            # completo | podado (trial de Optuna cortado por el pruner)
    "hiperparametros",
    "mae_cv", "std_cv", "n_folds",             # optuna_cv y baseline (TimeSeriesSplit sobre train)
    "mae_val", "rmse_val", "r2_val", "mae_train",
    "seg_entrenamiento", "n_features", "n_train", "n_val", "semilla", "dataset_sha256",
]


def registrar_experimento(fila: dict, ruta: str | Path = RUTA_LOGS) -> None:
    """Agrega una fila a results/experiment_logs.csv (lo crea si no existe).
    Si el log viene de una versión anterior del script (menos columnas), lo completa
    con las columnas nuevas sin perder las filas viejas."""
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    nueva = pd.DataFrame([{c: fila.get(c, np.nan) for c in COLUMNAS_LOG}])
    if not ruta.exists():
        nueva.to_csv(ruta, index=False)
        return
    if list(pd.read_csv(ruta, nrows=0).columns) == COLUMNAS_LOG:
        nueva.to_csv(ruta, mode="a", header=False, index=False)
    else:
        previo = pd.read_csv(ruta)
        extra = [c for c in previo.columns if c not in COLUMNAS_LOG]
        pd.concat([previo, nueva], ignore_index=True).reindex(columns=COLUMNAS_LOG + extra).to_csv(ruta, index=False)

## Optimización con Optuna

In [23]:
def _ajustar(clase: Callable, params: dict, X, y):
    params = dict(params)
    k = params.pop("k")
    kw = {n: params.pop(n) for n in ("umbral_redundancia",) if n in params}
    metodo_sel = params.pop("metodo_sel")
    pipe = construir_pipeline(clase(**params), X.columns, k, metodo_sel=metodo_sel)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return pipe.fit(X, y)


def _objetivo(esp: dict, X_tr: pd.DataFrame, y_tr: pd.Series, cv: TimeSeriesSplit,
              anotar: Callable) -> Callable[[optuna.Trial], float]:
    """Función objetivo: MAE medio sobre los folds de TimeSeriesSplit (solo train).
    Después de cada fold reporta el MAE medio acumulado; si el pruner decide que el trial
    no promete, se registra como 'podado' y se corta."""
    def objetivo(trial: optuna.Trial) -> float:
        metodos = ["spearman", "mutual_info"] + (["rfe"] if esp.get("rfe", True) else [])
        metodo = trial.suggest_categorical("metodo_sel", metodos)
        extra = {} if metodo == "rfe" else {"umbral_redundancia": trial.suggest_float("umbral_redundancia", 0.80, 0.99)}
        params = {**esp["fijos"], **esp["espacio"](trial),
                  "k": trial.suggest_int("k", 5, 150, step=5), "metodo_sel": metodo, **extra}
        maes: list[float] = []
        t0 = time.perf_counter()
        for paso, (i_fit, i_eval) in enumerate(cv.split(X_tr)):
            modelo = _ajustar(esp["clase"], params, X_tr.iloc[i_fit], y_tr.iloc[i_fit])
            maes.append(mean_absolute_error(y_tr.iloc[i_eval], modelo.predict(X_tr.iloc[i_eval])))
            trial.report(float(np.mean(maes)), step=paso)
            if trial.should_prune():
                anotar(params, maes, time.perf_counter() - t0, "podado")
                raise optuna.TrialPruned()
        anotar(params, maes, time.perf_counter() - t0, "completo")
        return float(np.mean(maes))
    return objetivo

### Comparaciones

In [24]:
def comparar_modelos(
    n_trials: int = 50,
    n_splits: int = 5,
    modelos: list[str] | None = None,
    timeout: float | None = None,
    ruta_datos: str | Path = RUTA_FINAL,
    ruta_logs: str | Path = RUTA_LOGS,
    verbose: bool = True,
) -> pd.DataFrame:
    """Baselines + un estudio de Optuna por modelo (TPE + MedianPruner, semilla fija).
    La búsqueda usa TimeSeriesSplit sobre train; la mejor configuración se reentrena en todo
    train y se mide en validation. Registra cada trial y cada modelo final en el log.
    Devuelve las filas de esta corrida."""
    modelos = modelos or list(ESPACIOS)
    desconocidos = set(modelos) - set(ESPACIOS)
    if desconocidos:
        raise ValueError(f"Modelos desconocidos: {sorted(desconocidos)}. Opciones: {list(ESPACIOS)}")

    df = cargar_datos(ruta_datos)  # sin test
    X_tr, y_tr = particion(df, "train")
    X_va, y_va = particion(df, "val")
    sha = huella(ruta_datos)[:12]
    corrida = datetime.now().isoformat(timespec="seconds")
    # test_size fijo: la ventana de entrenamiento más chica es la mitad de train. Con el default
    # el 1.er fold entrenaba con ~143 filas y 390 features y dominaba el MAE medio del CV.
    cv = TimeSeriesSplit(n_splits=n_splits, test_size=len(X_tr) // (2 * n_splits))
    filas = []

    def base_fila(modelo: str, etapa: str, estado: str, params: dict, seg: float) -> dict:
        return {
            "corrida": corrida, "timestamp": datetime.now().isoformat(timespec="seconds"),
            "modelo": modelo, "etapa": etapa, "estado": estado,
            "hiperparametros": json.dumps(params, sort_keys=True, default=str),
            "seg_entrenamiento": round(seg, 3),
            "n_features": X_tr.shape[1], "n_train": len(X_tr), "n_val": len(X_va),
            "semilla": SEMILLA, "dataset_sha256": sha,
        }

    def anotar_val(modelo: str, etapa: str, params: dict, m_val: dict, mae_train: float, seg: float,
                   extra: dict | None = None) -> None:
        # redondeo: con n_jobs=-1 el orden de suma de los árboles cambia el decimal 13
        # (ruido de punto flotante); a 6 decimales dos corridas dan exactamente lo mismo
        fila = {**base_fila(modelo, etapa, "completo", params, seg),
                "mae_val": round(m_val["mae"], 6), "rmse_val": round(m_val["rmse"], 6),
                "r2_val": round(m_val["r2"], 6), "mae_train": round(mae_train, 6), **(extra or {})}
        registrar_experimento(fila, ruta_logs)
        filas.append(fila)

    # --- baselines (mismas filas, mismo conjunto que los modelos). También se miden en los
    # mismos folds del CV: es la referencia para saber si un modelo le gana a la persistencia en CV.
    base = predicciones_baselines(df)
    for nombre in base:
        pred_tr = base.loc[X_tr.index, nombre]
        maes = [_metricas(y_tr.iloc[e], pred_tr.iloc[e])["mae"] for _, e in cv.split(X_tr)]
        cv_ref = {"mae_cv": round(float(np.mean(maes)), 6), "std_cv": round(float(np.std(maes)), 6),
                  "n_folds": len(maes)}
        anotar_val(nombre, "baseline", {}, _metricas(y_va, base.loc[X_va.index, nombre]),
                   _metricas(y_tr, pred_tr)["mae"], 0.0, cv_ref)
        if verbose:
            print(f"{nombre:26s} MAE cv = {cv_ref['mae_cv']:.1f}  MAE val = {filas[-1]['mae_val']:.1f}")

    # --- modelos
    for nombre in modelos:
        esp = ESPACIOS[nombre]
        t_modelo = time.perf_counter()

        def anotar_cv(params: dict, maes: list[float], seg: float, estado: str, _n=nombre) -> None:
            fila = {**base_fila(_n, "optuna_cv", estado, params, seg),
                    "mae_cv": round(float(np.mean(maes)), 6),
                    "std_cv": round(float(np.std(maes)), 6), "n_folds": len(maes)}
            registrar_experimento(fila, ruta_logs)
            filas.append(fila)

        estudio = optuna.create_study(
            direction="minimize",
            study_name=f"{nombre}_{corrida}",
            sampler=optuna.samplers.TPESampler(seed=SEMILLA),
            # no poda hasta tener 5 trials completos ni antes del 2.º fold
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        )
        estudio.optimize(_objetivo(esp, X_tr, y_tr, cv, anotar_cv),
                         n_trials=n_trials, timeout=timeout)

        # reconstruye los params de la mejor config (con el mapeo de capas del MLP)
        SEL = ("k", "metodo_sel", "umbral_redundancia")
        mejores = {**esp["fijos"], **esp["espacio"](optuna.trial.FixedTrial(estudio.best_params)),
                **{n: v for n, v in estudio.best_params.items() if n in SEL}}
        t0 = time.perf_counter()
        modelo = _ajustar(esp["clase"], mejores, X_tr, y_tr)
        seg = time.perf_counter() - t0
        anotar_val(nombre, "optuna_val", mejores, _metricas(y_va, modelo.predict(X_va)),
                   _metricas(y_tr, modelo.predict(X_tr))["mae"], seg)

        if verbose:
            estados = [t.state for t in estudio.trials]
            podados = sum(s == optuna.trial.TrialState.PRUNED for s in estados)
            print(f"{nombre:26s} MAE cv = {estudio.best_value:.1f}  MAE val = {filas[-1]['mae_val']:.1f}  "
                  f"({len(estados)} trials, {podados} podados, {time.perf_counter() - t_modelo:.0f} s)")

    return pd.DataFrame(filas)


def resumen(resultados: pd.DataFrame) -> pd.DataFrame:
    """Modelo final de cada estudio (y cada baseline), ordenado por MAE de validación."""
    finales = resultados[resultados["etapa"].isin(["baseline", "optuna_val"])]
    mejores = finales.loc[finales.groupby("modelo")["mae_val"].idxmin()]
    cv = (resultados[resultados["etapa"].isin(["optuna_cv", "baseline"])]
          .groupby("modelo")["mae_cv"].min().rename("mae_cv"))
    return (mejores.drop(columns="mae_cv", errors="ignore").join(cv, on="modelo")
            .sort_values("mae_val")
            [["modelo", "mae_cv", "mae_val", "rmse_val", "r2_val", "mae_train", "hiperparametros"]]
            .reset_index(drop=True))

## Pruebas

In [25]:
ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
ap.add_argument("--n-trials", type=int, default=50, help="trials de Optuna por modelo (default 50)")
ap.add_argument("--n-splits", type=int, default=5, help="folds de TimeSeriesSplit sobre train (default 5); cada fold de evaluación "
                     "mide len(train) // (2 * n_splits) días")
ap.add_argument("--modelos", type=str, default=None,
                help=f"lista separada por comas (default: todos = {','.join(ESPACIOS)})")
ap.add_argument("--timeout", type=float, default=None, help="segundos máximos por estudio")
args, _ = ap.parse_known_args(["--modelos", "lasso", "--n-trials", "5", "--n-splits", "3"])

RUTA_PRUEBA = ROOT / "results" / "logs_prueba.csv"

res = comparar_modelos(
    n_trials=args.n_trials, n_splits=args.n_splits, timeout=args.timeout,
    modelos=args.modelos.split(",") if args.modelos else None,
    ruta_logs=RUTA_PRUEBA,
)
print("\nModelo final por estudio (CV en train, medido en validación):")
print(resumen(res).drop(columns="hiperparametros").round(3).to_string(index=False))
print(f"\nLog: {RUTA_PRUEBA}")

baseline_persistencia      MAE cv = 3453.1  MAE val = 2691.8
baseline_media_movil_7d    MAE cv = 3579.6  MAE val = 2488.0
lasso                      MAE cv = 3788.5  MAE val = 2782.9  (5 trials, 0 podados, 692 s)

Modelo final por estudio (CV en train, medido en validación):
                 modelo   mae_cv  mae_val  rmse_val  r2_val  mae_train
baseline_media_movil_7d 3579.586 2487.983  3323.225   0.274   3407.849
  baseline_persistencia 3453.077 2691.758  3316.066   0.277   3310.523
                  lasso 3788.463 2782.897  3497.482   0.196   2801.167

Log: C:\Users\Axel\Desktop\cervecera_ml\results\logs_prueba.csv


In [26]:
from preprocess import columnas_elegidas

finales = res[res["etapa"] == "optuna_val"]
X_tr, y_tr = particion(cargar_datos(), "train")

for _, f in finales.iterrows():
    nombre = f["modelo"]
    params = json.loads(f["hiperparametros"])
    fijos = ESPACIOS[nombre]["fijos"]

    print(f"\n=== {nombre} | MAE val = {f['mae_val']:.1f}")
    print("Elegido por Optuna:", {k: v for k, v in params.items() if k not in fijos})

    # reajusta con esa config en todo train (lo mismo que se midió en validación)
    pipe = _ajustar(ESPACIOS[nombre]["clase"], params, X_tr, y_tr)
    cols = columnas_elegidas(pipe)
    print(f"Variables seleccionadas ({len(cols)}):")
    print(cols)


=== lasso | MAE val = 2782.9
Elegido por Optuna: {'alpha': 0.06026889128682508, 'k': 25, 'metodo_sel': 'mutual_info', 'umbral_redundancia': 0.9137451119974369}
Variables seleccionadas (25):
['.1', 'sala maq (kw)', 'frio (kw) mm3', 'frio (kw) mm14', 'planta (kw) mm7', 'planta (kw) mm14', 'meta et linea 5', 'servicios (kw)', 'unnamed: 3', 'planta (kw)', 'meta aire cocina', 'planta (kw) mm3', 'temp max (°c) x frio (kw)', 'mes_cos', 'kw cond 5. 6 y 9', 'kw cond 7. 8 y 11', 'agua planta (hl)', 'frio (kw) lag7', 'kw trafo 11', 'frio (kw) lag2', 'hl cerveza envasada mm14', 'planta (kw) lag7', 'et bodega/hl', 'kw mycom 7', 'temp max (°c) x hl cerveza envasada']
